# 04 Label Sentiment

This notebook creates the fixed, manually labelled sentiment-evaluation subset.

## Fixed sample

The submitted specification requires a fixed manually labelled subset but does not prescribe its size. The notebook selects 300 headlines: 30 from each of the ten companies. The fixed random seed `2025` makes the selection reproducible and company balancing prevents firms with more GDELT coverage from dominating the evaluation.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display

pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 200)

In [ ]:
# Support running the notebook from either the project root or pipeline directory.
working_directory = Path.cwd()
project_root = (
    working_directory.parent
    if working_directory.name == "pipeline"
    else working_directory
)

aligned_path = project_root / "data" / "processed" / "headlines_aligned_prices.csv"
gold_labels_path = project_root / "data" / "processed" / "gold_labels.csv"

print("Project root:", project_root)
print("Aligned data exists:", aligned_path.exists())

In [ ]:
aligned_df = pd.read_csv(aligned_path)

required_columns = {
    "headline_id",
    "published_at_utc",
    "published_at_london",
    "ticker",
    "company_name",
    "headline_text",
}

assert required_columns.issubset(aligned_df.columns)
assert aligned_df["headline_id"].is_unique
assert not aligned_df[list(required_columns)].isna().any().any()
assert aligned_df["ticker"].nunique() == 10

print(f"Available aligned headlines: {len(aligned_df):,}")

In [ ]:
SAMPLE_PER_COMPANY = 30
RANDOM_SEED = 2025

assert aligned_df.groupby("ticker").size().min() >= SAMPLE_PER_COMPANY

sample_df = (
    aligned_df.groupby("ticker", group_keys=False)
    .sample(n=SAMPLE_PER_COMPANY, random_state=RANDOM_SEED)
    .sort_values(["ticker", "published_at_utc", "headline_text"])
    .reset_index(drop=True)
)

# Replace the later member of each manually identified semantic-duplicate pair.
duplicate_replacements = {
    1: (
        "3d7dfe781decf4fc0114ae2198c293fd56e90d7676b91e46f7e444872358920d",
        "b5fea3607bcfdf442b99b9975d611804dcb7faeb95ee472def5b9ac095a7bb1d",
    ),
    143: (
        "a6d2872810ea241f7df20cb4bee1039a48a5cbb9bc6d19aaaffcbfdb6d803a94",
        "160526a9b5332786c0bb64ce308c483fa9ceaec05eb5d9f6da5f817ce0deb192",
    ),
    163: (
        "38dc6600edaedb80c9b7919cb52d914f74646efe40b4be699be8ed50380475c1",
        "cb1a4838296bbdcdda312669fc261cd71e5d2a3a9a4c1102a69922dcd251f025",
    ),
}

for row_number, (duplicate_id, replacement_id) in duplicate_replacements.items():
    assert sample_df.loc[row_number, "headline_id"] == duplicate_id
    assert replacement_id not in set(sample_df["headline_id"])

    replacement = aligned_df.loc[aligned_df["headline_id"] == replacement_id]
    assert len(replacement) == 1
    assert replacement.iloc[0]["ticker"] == sample_df.loc[row_number, "ticker"]

    sample_df.loc[row_number] = replacement.iloc[0][sample_df.columns]

assert len(sample_df) == 300
assert sample_df["headline_id"].is_unique
assert sample_df.groupby("ticker").size().eq(SAMPLE_PER_COMPANY).all()

display(sample_df.groupby("ticker").size().rename("sampled_headlines"))

## Labelling guide

Labels were assigned according to the likely financial effect on the named target company, based solely on the headline. Subsequent share-price movements, article bodies, and external knowledge were not considered.

- **positive**: a clearly favourable development, such as improved results, an approval, a contract win, or an upgrade.
- **negative**: a clearly unfavourable development, such as weaker results, a warning, litigation, disruption, or a downgrade.
- **neutral**: factual, mixed, unclear, or without a clear positive or negative effect.

This headline-only, entity-aware approach follows [Sinha et al. (2022)](https://doi.org/10.1002/asi.24634).

In [ ]:
display_columns = [
    "headline_id",
    "ticker",
    "company_name",
    "published_at_london",
    "headline_text",
]

labelling_df = sample_df[display_columns].copy()
labelling_df["label_gold"] = ""
labelling_df["annotator_pass"] = 1

valid_labels = {"positive", "neutral", "negative"}


def assign_labels(indices, label):
    """Assign one final sentiment label to the specified sample rows."""

    assert label in valid_labels
    assert labelling_df.loc[indices, "label_gold"].eq("").all()
    labelling_df.loc[indices, "label_gold"] = label

In [ ]:
# AstraZeneca (AZN.L): rows 0-29
assign_labels([0, 1, 2, 6, 9, 16, 17, 18, 19, 23, 25, 27], "positive")
assign_labels([3, 4, 7, 8, 15, 24], "negative")
assign_labels([5, 10, 11, 12, 13, 14, 20, 21, 22, 26, 28, 29], "neutral")

In [ ]:
# GSK (GSK.L): rows 30-59
assign_labels(
    [31, 33, 34, 35, 36, 38, 41, 43, 44, 45, 46, 47, 50, 52, 56, 57],
    "positive",
)
assign_labels([39, 40, 42, 49], "negative")
assign_labels([30, 32, 37, 48, 51, 53, 54, 55, 58, 59], "neutral")

In [ ]:
# Halma (HLMA.L): rows 60-89
assign_labels(
    [64, 65, 66, 67, 68, 69, 70, 72, 73, 74, 75, 76, 77, 79, 80, 81, 82, 83, 84, 88],
    "positive",
)
assign_labels([62, 78, 85], "negative")
assign_labels([60, 61, 63, 71, 86, 87, 89], "neutral")

In [ ]:
# HSBC (HSBA.L): rows 90-119
assign_labels([90, 92, 93, 96, 97, 100, 103, 111, 113, 114, 115], "positive")
assign_labels([94, 95, 99, 102, 109], "negative")
assign_labels(
    [91, 98, 101, 104, 105, 106, 107, 108, 110, 112, 116, 117, 118, 119],
    "neutral",
)

In [ ]:
# Intertek (ITRK.L): rows 120-149
assign_labels(
    [120, 121, 122, 123, 124, 125, 126, 130, 132, 133, 135, 136, 137, 141, 142, 145, 146, 147, 148],
    "positive",
)
assign_labels([128, 129, 134, 140, 144], "negative")
assign_labels([127, 131, 138, 139, 143, 149], "neutral")

In [ ]:
# National Grid (NG.L): rows 150-179
assign_labels([155, 162, 165, 170, 171, 173, 176, 177, 179], "positive")
assign_labels([150, 152, 153, 157, 158, 159, 164, 166, 167, 172, 178], "negative")
assign_labels([151, 154, 156, 160, 161, 163, 168, 169, 174, 175], "neutral")

In [ ]:
# RELX (REL.L): rows 180-209
assign_labels(
    [180, 181, 182, 183, 184, 187, 189, 190, 191, 192, 194, 195, 196, 201, 202, 203, 204, 206, 207, 208],
    "positive",
)
assign_labels([185, 186, 193, 197, 209], "negative")
assign_labels([188, 198, 199, 200, 205], "neutral")

In [ ]:
# Rio Tinto (RIO.L): rows 210-239
assign_labels([215, 216, 219, 220, 221, 225, 227, 230, 232, 233, 237], "positive")
assign_labels([210, 217, 218, 223, 224, 226, 228, 236], "negative")
assign_labels([211, 212, 213, 214, 222, 229, 231, 234, 235, 238, 239], "neutral")

In [ ]:
# Smith & Nephew (SN.L): rows 240-269
assign_labels(
    [240, 241, 242, 243, 244, 246, 247, 248, 253, 259, 260, 261, 262, 264, 265, 266, 267, 268],
    "positive",
)
assign_labels([249, 252, 254, 269], "negative")
assign_labels([245, 250, 251, 255, 256, 257, 258, 263], "neutral")

In [ ]:
# Vodafone (VOD.L): rows 270-299
assign_labels([271, 273, 275, 283, 285, 286, 287, 288, 289, 294, 296, 297], "positive")
assign_labels([272, 274, 276, 280, 281, 282, 291, 292, 295, 298], "negative")
assign_labels([270, 277, 278, 279, 284, 290, 293, 299], "neutral")

In [ ]:
assert len(labelling_df) == 300
assert labelling_df["headline_id"].is_unique
assert labelling_df.groupby("ticker").size().eq(SAMPLE_PER_COMPANY).all()
assert set(labelling_df["label_gold"]) == valid_labels
assert not labelling_df["label_gold"].eq("").any()
assert labelling_df["annotator_pass"].eq(1).all()

gold_columns = ["headline_id", "label_gold", "annotator_pass"]
gold_labels_df = labelling_df[gold_columns].copy()

gold_labels_path.parent.mkdir(parents=True, exist_ok=True)
gold_labels_df.to_csv(gold_labels_path, index=False)

display(gold_labels_df["label_gold"].value_counts())
print(f"Saved {len(gold_labels_df)} final labels to {gold_labels_path}")